# Parallel Computation (Timing)

This notebook measures the efficiency gains when using parallelization during simulation. For the description of how LightCurveLynx parallelizes computation, along with technical details, see the [parallelization notebook](https://lightcurvelynx.readthedocs.io/en/latest/notebooks/parallel_runs.html).

We run multiple simulations of [sncomso](https://sncosmo.readthedocs.io/en/stable/)'s SALT2 model with varying levels of parallelization.

## Input Data

For the simulation we use a realistic survey cadence and set of passbands for LSST. For the survey we pre-download the v5.1.1 simulated survey from [https://s3df.slac.stanford.edu/data/rubin/sim-data/](https://s3df.slac.stanford.edu/data/rubin/sim-data/). For filters we use the ones defined by the LSST preset. Both are stored in a local "data" directory.

In [ ]:
from lightcurvelynx.astro_utils.passbands import PassbandGroup
from lightcurvelynx.obstable.opsim import OpSim

opsim_file = "../data/opsim/baseline_v5.1.1_10yrs.db"
ops_data = OpSim.from_db(opsim_file)
print(f"Loaded OpSim data with {len(ops_data)} entries.")

t_min, t_max = ops_data.time_bounds()
print(f"Time bounds of OpSim data: {t_min} to {t_max}")

table_dir = "../data/passbands/LSST"
passband_group = PassbandGroup.from_preset(preset="LSST", table_dir=table_dir)
print(f"Loaded passband group with {len(passband_group)} passbands.")

## Model

For the object model, we use a SALT2 model (sncosmo's "salt2-h17) with a position that is randomly drawn from the coverage of the survey and a `t0` that is drawn uniformly from the survey's time coverage. The remaining parameters are set as constants with: `redshift`=0.1, `x0`=1.0e-6, `x1`=0.5, and `c`=0.2.

Since the sncosmo models are only defined for a limited time window around t0, we extrapolate with a linear decay to zero over 50 days.

In [ ]:
from lightcurvelynx.math_nodes.np_random import NumpyRandomFunc
from lightcurvelynx.math_nodes.ra_dec_sampler import ApproximateMOCSampler
from lightcurvelynx.models.sncosmo_models import SncosmoWrapperModel
from lightcurvelynx.utils.extrapolate import LinearDecay

ra_dec_sampler = ApproximateMOCSampler.from_obstable(ops_data, depth=6)

source = SncosmoWrapperModel(
    "salt2-h17",
    t0=NumpyRandomFunc("uniform", low=t_min, high=t_max),
    x0=1.0e-6,
    x1=0.5,
    c=0.2,
    ra=ra_dec_sampler.ra,
    dec=ra_dec_sampler.dec,
    redshift=0.01,
    node_label="source",
    time_extrapolation=LinearDecay(decay_width=50.0),
)

## Simulation

We perform a basic parallelization using the (default) `ProcessPoolExecutor`. Note that this approach will have significant overhead per worker as it copies all the state.

In [ ]:
import functools
import timeit
import numpy as np

from lightcurvelynx.simulate import simulate_lightcurves

num_samples_list = [50_000]
num_jobs_list = [1, 2, 3, 4]
num_simulations = 10

all_times = np.zeros((len(num_samples_list), len(num_jobs_list), num_simulations))
for sample_idx, num_samples in enumerate(num_samples_list):
    for job_idx, num_jobs in enumerate(num_jobs_list):
        print(f"Starting {num_simulations} simulations of {num_samples} samples with num_jobs={num_jobs}:")
        simulate_func = functools.partial(
            simulate_lightcurves,
            model=source,
            num_samples=num_samples,
            obstable=ops_data,
            passbands=passband_group,
            num_jobs=num_jobs,
            obs_time_window_offset=(-100, 400),
        )

        for trial in range(num_simulations):
            print(f"  Simulation {trial + 1}/{num_simulations} with num_jobs={num_jobs}:")
            t = timeit.timeit(simulate_func, number=1)
            all_times[sample_idx, job_idx, trial] = t
            print(f"  Time for this simulation: {t:.2f} seconds")

        total_time = np.sum(all_times[sample_idx, job_idx])
        average_time = np.mean(all_times[sample_idx, job_idx])
        print(f"Time (sec) for num_jobs={num_jobs}: Total={total_time:.2f}. Average={average_time:.2f}")
        print("-" * 50)

print(all_times)

In [ ]:
all_times = np.array(
    [
        [
            [
                23.81536179,
                25.07706879,
                24.42103271,
                24.52286629,
                23.08002613,
                22.88664421,
                22.79947325,
                23.23794833,
                23.73173042,
                23.86884604,
            ],
            [
                19.362552,
                19.742056,
                18.05812833,
                17.38001829,
                18.67572125,
                19.04033042,
                18.36831338,
                18.26862571,
                18.75648917,
                18.86241608,
            ],
            [
                21.39083371,
                19.22883967,
                18.49785233,
                18.82757104,
                20.23626304,
                22.100096,
                18.92257129,
                19.42681508,
                18.23460817,
                17.38817121,
            ],
            [
                20.84591983,
                20.97791321,
                21.07696479,
                22.51660783,
                21.43930233,
                20.20130642,
                19.28660388,
                21.10406787,
                21.90376229,
                20.73969375,
            ],
        ],  # 10_000 samples
        [
            [
                44.24531767,
                43.05006025,
                43.13663221,
                42.56546171,
                42.39540008,
                43.48928404,
                42.04287767,
                44.70132979,
                42.14325304,
                43.67275875,
            ],
            [
                27.52842725,
                27.9038495,
                27.95149662,
                28.51042525,
                30.63098062,
                30.76627638,
                29.18122642,
                29.53554342,
                29.66724004,
                30.58296229,
            ],
            [
                30.66742996,
                25.17899379,
                31.26397308,
                26.28668033,
                25.42611596,
                27.22581838,
                28.7612965,
                25.14675038,
                32.85701467,
                25.28889188,
            ],
            [
                40.959584,
                39.69906075,
                44.08133629,
                43.35491596,
                42.86800942,
                46.28901421,
                46.52616108,
                39.82764379,
                45.47412558,
                42.64075275,
            ],
        ],  # 20_000 samples
        [
            [
                64.21115433,
                63.43778104,
                65.57528838,
                64.87636179,
                64.58633496,
                65.06382617,
                65.00557675,
                64.64833133,
                65.43010429,
                63.3821875,
            ],
            [
                39.3586175,
                39.72825587,
                39.18173725,
                41.18811867,
                38.80658379,
                38.88423133,
                39.53485737,
                39.84855108,
                39.78940742,
                41.08267062,
            ],
            [
                36.26197233,
                33.48404604,
                34.06746783,
                39.14263692,
                36.18781433,
                35.80852496,
                35.25844758,
                36.51988487,
                34.96532625,
                36.38305887,
            ],
            [
                49.77084288,
                44.64442862,
                48.34326929,
                40.96355408,
                44.62846063,
                45.7329095,
                38.73624475,
                41.04307517,
                40.67945221,
                39.70884513,
            ],
        ],  # 30_000 samples
        [
            [
                79.60828767,
                80.006632,
                78.9264765,
                80.57966092,
                81.83866354,
                80.01674175,
                78.37819346,
                79.03848317,
                79.2989705,
                79.25249596,
            ],
            [
                47.23946458,
                47.69754571,
                47.68921875,
                46.74953408,
                48.72543892,
                47.33140471,
                47.98749196,
                46.84961842,
                46.25275654,
                47.84466871,
            ],
            [
                42.13981817,
                39.36095112,
                40.34755867,
                38.01884704,
                40.73007167,
                41.80728642,
                51.74916579,
                45.824291,
                40.82170958,
                39.06407229,
            ],
            [
                55.80552925,
                59.96286696,
                59.67492154,
                65.67323921,
                52.365813,
                68.65157596,
                60.34458579,
                92.9095945,
                58.23054779,
                73.92916412,
            ],
        ],  # 40_000 samples
        [
            [
                102.81764975,
                99.89414971,
                97.76066421,
                99.849167,
                101.95069088,
                98.79291196,
                97.61702546,
                97.43718142,
                98.44278721,
                98.32666708,
            ],
            [
                58.17175338,
                58.84284721,
                55.22870879,
                59.11886071,
                57.95794371,
                57.77686117,
                57.10540433,
                55.190418,
                57.38950908,
                57.58867092,
            ],
            [
                49.29301325,
                49.22912392,
                47.27887079,
                49.6636495,
                55.16364829,
                51.94023071,
                47.54114,
                45.70964613,
                53.39096308,
                48.61895379,
            ],
            [
                83.28768079,
                96.20273996,
                73.96163513,
                93.68481092,
                71.13517742,
                110.76274463,
                87.88426708,
                73.12435,
                115.91326192,
                62.27633408,
            ],
        ],
    ]
)
num_samples_list = [10_000, 20_000, 30_000, 40_000, 50_000]

In [ ]:
import matplotlib.pyplot as plt

fix, ax = plt.subplots(figsize=(10, 6))
for jobs_idx, num_jobs in enumerate(num_jobs_list):
    ave_times = np.mean(all_times[:, jobs_idx, :], axis=1)
    ax.plot(num_samples_list, ave_times, label=f"num_jobs={num_jobs}")

ax.set_xlabel("Number of Samples")
ax.set_ylabel("Average Time (seconds)")
ax.set_title("Parallel Simulation Timing")
ax.legend()
plt.show()

In [ ]:
fix, ax = plt.subplots(figsize=(10, 6))
for sample_idx, num_samples in enumerate(num_samples_list):
    ave_times = np.mean(all_times[sample_idx, :, :], axis=1)

    # Compute the 95% confidence interval on times.
    std_times = np.std(all_times[sample_idx, :, :], axis=1)
    ci = 1.96 * std_times / np.sqrt(num_simulations)

    plt.errorbar(num_jobs_list, ave_times, yerr=ci, label=f"num_samples={num_samples}", capsize=5, marker="o")

ax.set_xlabel("Number of Jobs")
ax.set_ylabel("Average Time (seconds)")
ax.set_title("Parallel Simulation Timing")
ax.legend()
plt.show()